In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torchvision.io import decode_image
import os
from PIL import Image
import pandas as pd

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
zip_url="https://github.com/ishananand06/CSOT26_Computer-Vision/raw/2fd0d1ce24662f1cc9f5c0ad4d36c2163a22d973/Week%201/test_set.zip"
if os.path.exists('test_set.zip'):
    os.remove('test_set.zip')

os.makedirs('FashionMNIST/test_files', exist_ok=True)
!wget {zip_url} -O test_set.zip
!unzip -o test_set.zip -d FashionMNIST/test_files

--2026-06-05 17:44:30--  https://github.com/ishananand06/CSOT26_Computer-Vision/raw/2fd0d1ce24662f1cc9f5c0ad4d36c2163a22d973/Week%201/test_set.zip
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ishananand06/CSOT26_Computer-Vision/2fd0d1ce24662f1cc9f5c0ad4d36c2163a22d973/Week%201/test_set.zip [following]
--2026-06-05 17:44:30--  https://raw.githubusercontent.com/ishananand06/CSOT26_Computer-Vision/2fd0d1ce24662f1cc9f5c0ad4d36c2163a22d973/Week%201/test_set.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 103725 (101K) [application/zip]
Saving to: ‘test_set.zip’

test_set.zip        100%[======

In [4]:
class InferenceImageDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.img_names = [f for f in os.listdir(img_dir)]

    def __len__(self):
        return len(self.img_names)
    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path)
        if self.transform:
            image = self.transform(image)
        return image, img_name

In [5]:
test_transforms = v2.Compose([v2.ToImage(),
    v2.Resize((28,28)),
    v2.ToDtype(torch.float32,scale=True)
])
test_dir = "FashionMNIST/test_files/images"
test_dataset = InferenceImageDataset(img_dir=test_dir,transform=test_transforms)
test_inf_dataloader=DataLoader(test_dataset, batch_size=100, shuffle=False)

In [6]:
training_data=datasets.FashionMNIST(root="FashionMNIST",train=True,
transform=v2.Compose([v2.ToImage(),v2.RandomHorizontalFlip(p=0.5),v2.ToDtype(torch.float32,scale=True)]),download=True)

test_data=datasets.FashionMNIST(root="FashionMNIST",train=False,
transform=v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32,scale=True)]),download=True)



100%|██████████| 26.4M/26.4M [00:00<00:00, 115MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 3.89MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 54.0MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 18.4MB/s]


In [7]:
batch_size=100
training_dataloader=DataLoader(training_data,batch_size=batch_size,shuffle=True)
test_dataloader=DataLoader(test_data,batch_size=batch_size,shuffle=False)

In [8]:
class neural_network(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten=nn.Flatten()
    self.linear=nn.Sequential(
        nn.Linear(784,512),
        nn.BatchNorm1d(512),
        nn.LeakyReLU(0.1),
        nn.Dropout(0.1),
        nn.Linear(512,256),
        nn.BatchNorm1d(256),
        nn.LeakyReLU(0.1),
        nn.Dropout(0.1),
        nn.Linear(256,10)
    )
  def forward(self,x):
    x=self.flatten(x)
    logits=self.linear(x)
    return logits
model=neural_network()
model = model.to(device)

In [9]:
learning_rate=0.001
num_epochs=20
loss_fn=nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4, amsgrad=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

In [10]:
def train_loop(dataloader,model,loss_fn,optimizer):
  model.train()
  size=len(dataloader.dataset)
  correct=0
  for batch,(X,y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)
    pred=model(X)
    loss=loss_fn(pred,y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if batch%100==0:
      loss_val=loss.item()
      current=batch*batch_size+len(X)
      print(f"loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")
    correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    optimizer.zero_grad()
  train_accuracy = correct / size
  print(f"Training Accuracy: {(100*train_accuracy):>0.1f}%")

def test_loop(dataloader,model,loss_fn):
  model.eval()
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  test_loss, correct = 0, 0
  with torch.no_grad():
    for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred=model(X)
            test_loss+=loss_fn(pred,y).item()
            correct+=(pred.argmax(1)==y).type(torch.float).sum().item()
  test_loss/=num_batches
  correct/=size
  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
  return test_loss

In [11]:
for t in range(num_epochs):
    print(f"Epoch {t+1}")
    train_loop(training_dataloader,model,loss_fn,optimizer)
    vloss=test_loop(test_dataloader,model,loss_fn)
    scheduler.step(vloss)


Epoch 1
loss: 2.458834  [  100/60000]
loss: 0.423239  [10100/60000]
loss: 0.396418  [20100/60000]
loss: 0.382435  [30100/60000]
loss: 0.385010  [40100/60000]
loss: 0.434432  [50100/60000]
Training Accuracy: 82.7%
Test Error: 
 Accuracy: 84.8%, Avg loss: 0.426148 

Epoch 2
loss: 0.399156  [  100/60000]
loss: 0.340822  [10100/60000]
loss: 0.248497  [20100/60000]
loss: 0.331396  [30100/60000]
loss: 0.373534  [40100/60000]
loss: 0.401177  [50100/60000]
Training Accuracy: 86.2%
Test Error: 
 Accuracy: 85.9%, Avg loss: 0.382883 

Epoch 3
loss: 0.352333  [  100/60000]
loss: 0.287123  [10100/60000]
loss: 0.267291  [20100/60000]
loss: 0.266638  [30100/60000]
loss: 0.392820  [40100/60000]
loss: 0.278298  [50100/60000]
Training Accuracy: 87.3%
Test Error: 
 Accuracy: 86.9%, Avg loss: 0.366099 

Epoch 4
loss: 0.357055  [  100/60000]
loss: 0.322454  [10100/60000]
loss: 0.143646  [20100/60000]
loss: 0.306498  [30100/60000]
loss: 0.426937  [40100/60000]
loss: 0.233630  [50100/60000]
Training Accuracy

In [12]:

model.eval()
image_ids=[]
predictions=[]
with torch.no_grad():
    for images, img_names in test_inf_dataloader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, dim=1)
        image_ids.extend(img_names)
        predictions.extend(preds.cpu().numpy())
print(f"Finished predicting")

Finished predicting


In [13]:
results=pd.DataFrame({
    'image_id':image_ids,
    'label':predictions
})
results=results.sort_values(by='image_id').reset_index(drop=True)
results.to_csv('2025MT11352_NN.csv', index=False)
print("predictions successfully saved ")

predictions successfully saved 
